In [1]:
import numpy as np
import pandas as pd
import os 
from pathlib import Path
print("현재 위치:", os.getcwd())

현재 위치: /Users/yuri.seo/Documents/GitHub/webnovel-pass-ab-test/notebooks


In [2]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"


coin_transactions = pd.read_csv(DATA / "coin_transactions.csv.gz",parse_dates=['txn_at'])
daily_reading = pd.read_csv(DATA / "daily_reading.csv.gz",parse_dates = ['read_date'])
exposures = pd.read_csv(DATA / "exposures.csv.gz",parse_dates = ['exposed_at'])
subscription_events = pd.read_csv(DATA / "subscription_events.csv",parse_dates = ['event_at'])
titles = pd.read_csv(DATA / "titles.csv")
users = pd.read_csv(DATA / "users.csv.gz",parse_dates = ['signup_date'])


# 각 테이블 정보 확인을 위해 딕셔너리 생성
table_info = {"coin_transactions":coin_transactions,
      "daily_reading":daily_reading,
     "exposures":exposures,
     "subscription_events":subscription_events,
     "titles":titles,
     "users":users}

for key, val in table_info.items():
    print(key, val.shape)

coin_transactions (825552, 6)
daily_reading (1865128, 6)
exposures (315122, 4)
subscription_events (61011, 5)
titles (6300, 4)
users (315200, 9)


In [3]:
# 데이터 탐색

# 범주형 컬럼 별 값이 몇개 있는지 확인 
print(exposures["variant"].value_counts()) # A 157567 B 157555
print(exposures["platform"].value_counts()) # app 267657 web 47465
print(subscription_events["event_type"].value_counts()) # trial_start 26406 trial_cancel 18607 payment 13363 cancel 2635 
print(coin_transactions["purpose"].value_counts()) 
print(daily_reading["access_type"].value_counts())
print(titles["genre"].value_counts())
print(titles["status"].value_counts())
print(users["age_group"].value_counts())
print(users["acquisition_channel"].value_counts())
print(users["preferred_genre"].value_counts())



variant
A    157567
B    157555
Name: count, dtype: int64
platform
app    267657
web     47465
Name: count, dtype: int64
event_type
trial_start     26406
trial_cancel    18607
payment         13363
cancel           2635
Name: count, dtype: int64
purpose
latest_episode       406815
completed_episode    265719
skip_wait            153018
Name: count, dtype: int64
access_type
wait_free    637831
coin         611114
pass         397191
free         218992
Name: count, dtype: int64
genre
로맨스판타지    2284
현대판타지     1558
로맨스        942
무협         910
기타         606
Name: count, dtype: int64
status
completed    4800
serial       1500
Name: count, dtype: int64
age_group
20대       131210
30대        94944
40대 이상     51818
10대        37228
Name: count, dtype: int64
acquisition_channel
organic    170177
paid_ad     91901
sns         46729
Name: count, dtype: int64
preferred_genre
로맨스판타지    110298
현대판타지      86057
무협         49392
로맨스        42584
기타         26869
Name: count, dtype: int64


In [4]:
print('\n',"결측치 검증")
for key, val in table_info.items():
    print(key, val.isna().sum())



 결측치 검증
coin_transactions txn_id      0
user_id     0
title_id    0
txn_at      0
coins       0
purpose     0
dtype: int64
daily_reading user_id            0
read_date          0
access_type        0
serial_reads       0
completed_reads    0
titles_finished    0
dtype: int64
exposures user_id       0
variant       0
exposed_at    0
platform      0
dtype: int64
subscription_events event_id      0
user_id       0
event_type    0
event_at      0
amount_krw    0
dtype: int64
titles title_id          0
genre             0
status            0
total_episodes    0
dtype: int64
users user_id                     0
signup_date                 0
age_group                   0
acquisition_channel      6393
preferred_genre             0
pre30_serial_reads          0
pre30_completed_reads       0
pre30_free_reads            0
pre30_coin_spend            0
dtype: int64


In [5]:

print('\n',"PK 중복 확인")
print(users["user_id"].duplicated().sum())
print(titles["title_id"].duplicated().sum())
print(coin_transactions["txn_id"].duplicated().sum())
print(subscription_events["event_id"].duplicated().sum())
print(exposures["user_id"].duplicated().sum())
print(daily_reading.duplicated(subset=["user_id","read_date","access_type"]).sum())




 PK 중복 확인
0
0
0
0
0
0


In [6]:

#describe
print("최댓값, 최소값 확인")
print(coin_transactions["coins"].describe())       #mean1.84 std 2.092 min 1.0, 75% 2.0 max 100 -> max값이 수상함. 확인필요
print(daily_reading["completed_reads"].describe())
print(subscription_events["amount_krw"].describe())


최댓값, 최소값 확인
count    825552.000000
mean          1.847013
std           2.027420
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         100.000000
Name: coins, dtype: float64
count    1.865128e+06
mean     2.686754e+00
std      6.148735e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.000000e+00
max      1.190000e+02
Name: completed_reads, dtype: float64
count    61011.000000
mean      2168.358165
std       4094.538251
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       9900.000000
Name: amount_krw, dtype: float64


In [7]:
coin_transactions["user_id"].loc[coin_transactions["coins"]==100].nunique()  ## 200
coin100 = set(coin_transactions["user_id"].loc[coin_transactions["coins"]==100] )          ## test_0NNN 리스트 200개 확인됨. test계정이 섞여있음.
cointest = set(coin_transactions["user_id"].loc[coin_transactions["user_id"].str.startswith("test_")] )
##test계정을 없애자. 
## users.loc[users["user_id"].str.startswith('test')].nunique()
coin100 == cointest ## coin을 100개 쓴 사람은 test계정뿐

## users테이블에서 테스트계정 리스트 뽑기
test_user =  users["user_id"].loc[users["user_id"].str.startswith('test_')]
#test_user.isin(cointest).value_counts() #확인 OK



In [8]:

#테스트계정 삭제하기. (200행 제거) 

is_test = coin_transactions["user_id"].isin(test_user) # test계정 확인 
keep = ~is_test #남겨둘 것 (test계정이 아니면 True)
coin_clean = coin_transactions.loc[keep] 
print(len(coin_transactions), len(coin_clean), len(coin_transactions) - len(coin_clean))


# 다른 테이블도 동일하게 없애기
for name , df in table_info.items():
    if name == "titles":
        continue

    before = len(df)
    df_clean = df.loc[~df["user_id"].isin(test_user)]
    after = len(df_clean)

    table_info[name] = df_clean
    print(f"{name}: {before:,} → {after:,} ({before - after:,}행 제거)")

users = table_info["users"]
coin_transactions = table_info["coin_transactions"]
subscription_events = table_info["subscription_events"]
exposures = table_info["exposures"]
daily_reading = table_info["daily_reading"]



825552 825352 200
coin_transactions: 825,552 → 825,352 (200행 제거)
daily_reading: 1,865,128 → 1,864,928 (200행 제거)
exposures: 315,122 → 314,922 (200행 제거)
subscription_events: 61,011 → 60,611 (400행 제거)
users: 315,200 → 315,000 (200행 제거)


In [9]:
### event_id 중복 확인

print(subscription_events.duplicated(subset = ["user_id","event_type","event_at","amount_krw"]).value_counts()) # TRUE(중복) 66

#중복행만 거르기
subscription_events.loc[subscription_events.duplicated(subset = ["user_id","event_type","event_at","amount_krw"],
                                                       keep =False ),
    'event_type'
    ].value_counts()
# payment에서만 중복 발생
subscription_events.loc[subscription_events.duplicated(subset = ["user_id","event_type","event_at","amount_krw"],
                                                       keep =False )]


# 유저 한명 확인
subscription_events.loc[subscription_events["user_id"] == "U225096"]  ## 같은시간에 중복으로 찍혀있는 데이터 발견 

# 지우기 
table_info["subscription_events"] = subscription_events.drop_duplicates(subset = ["user_id","event_type","event_at","amount_krw"],
                                                                       keep = 'first') 
subscription_events = table_info["subscription_events"]

print(subscription_events.duplicated(subset = ["user_id","event_type","event_at","amount_krw"]).value_counts())

False    60545
True        66
Name: count, dtype: int64
False    60545
Name: count, dtype: int64


In [10]:
## 노출 전에 trial시작한 유저가 있는지 

# trial시작한 유저 찾기
trial_user =  table_info["subscription_events"].loc[table_info["subscription_events"]["event_type"]=="trial_start","user_id"]
# 노출 유저 수
print("전체유저:", table_info["exposures"]["user_id"].nunique()) ##314,922


## 소개화면 노출 유저 unique
exposed_user = set(table_info["exposures"]["user_id"])


trial_user.isin(exposed_user).value_counts()    # trial유저 중 소개 화면 노출 유저 검증 
                                                # TRUE 26128 False78 

## lost user list 확인
lost_user = table_info["subscription_events"].loc[~table_info["subscription_events"]["user_id"].isin(exposed_user),"user_id"].unique()

## Lost User의 경우 A/B variant구분이 어렵고 전체 유저 중 0.2%비율로 매우 낮을 것으로 판단. 매출 확인 후 액션 판단

# Lost User 중 Payment이력이 있는 유저를 찾아보자
events =  table_info["subscription_events"]
lost_payment =  events["amount_krw"].loc[(events["user_id"].isin(lost_user))&(events["event_type"]=="payment")].sum()
print(lost_payment)
total_payment = events["amount_krw"].sum()
print(lost_payment, total_payment, lost_payment / total_payment) # 316800 129660300 0.002443307627701  


# Lost User 제외하기

for name, df in table_info.items():
    if name == "titles":
        continue

    before = len(df)
    df_clean = df.loc[~df["user_id"].isin(lost_user)]
    after = len(df_clean)

    table_info[name] = df_clean
    print(f"{name}: {before:,} → {after:,} ({before - after:,}행 제거)")


users = table_info["users"]
coin_transactions = table_info["coin_transactions"]
subscription_events = table_info["subscription_events"]
exposures = table_info["exposures"]
daily_reading = table_info["daily_reading"]



전체유저: 314922
316800
316800 129660300 0.002443307627701
coin_transactions: 825,352 → 823,238 (2,114행 제거)
daily_reading: 1,864,928 → 1,859,866 (5,062행 제거)
exposures: 314,922 → 314,922 (0행 제거)
subscription_events: 60,545 → 60,367 (178행 제거)
users: 315,000 → 314,922 (78행 제거)


발견: 체험 시작자 중 노출 기록 없는 유저 78명 (0.30%)  
영향 규모: 결제액 비중 0.24%, 전환율 ○○% (전체 약 29%와 유사)  
처리: 전 테이블에서 제외  
근거: ① 실험군(A/B) 정보가 없어 비교 분석에 포함할 수 없음 ② 유저·매출 비중이 작고 행동이 전체와 비슷해 제외해도 결과 왜곡이 작음 

In [11]:
# 결측 확인

print(table_info["users"]["acquisition_channel"].isna().sum())

user_t = table_info["users"]

channel_isna_user = set(user_t.loc[user_t["acquisition_channel"].isna()]["user_id"]) #결측 유저 아이디

#결측 유저 아이디의 variant 비중 보기
exposures_t =  table_info["exposures"]
isna_v = exposures_t.loc[exposures_t["user_id"].isin(channel_isna_user)][["user_id","variant"]]
isna_v["variant"].value_counts(normalize=True) #A 50.4% B 49.59% 

# 결측치 채우기. 비율 비슷하므로 'unknown'으로 채우자
table_info["users"]["acquisition_channel"] = table_info["users"]["acquisition_channel"].fillna('unknown')
users = table_info["users"]


6388


In [12]:
# ── 이슈별 정제 내역 ──
# 이슈 하나 = 딕셔너리 하나, 4개를 리스트로 묶어서 DataFrame으로 변환
cleaning_log = pd.DataFrame([
    {"이슈": "내부 테스트 계정",
     "대상 테이블": "titles 제외 전체",
     "규모": f"{len(test_user):,}명",
     "처리": "전 테이블에서 제거",
     "근거": "test_ 접두 ID, 체험 5분 내 결제·100코인 일괄 사용 등 비정상 행동"},

    {"이슈": "중복 결제 이벤트",
     "대상 테이블": "subscription_events",
     "규모": "66건",
     "처리": "event_id 외 동일 행 중 첫 행만 유지",
     "근거": "동일 유저·시각·금액 결제가 2회 기록 (로그 중복)"},

    {"이슈": "노출 기록 유실",
     "대상 테이블": "titles 제외 전체",
     "규모": f"{len(lost_user):,}명",
     "처리": "전 테이블에서 제거",
     "근거": "실험군(A/B) 확인 불가, 체험 시작자의 0.30%·결제액의 0.24%로 영향 미미"},

    {"이슈": "유입 경로 결측",
     "대상 테이블": "users",
     "규모": f"{len(channel_isna_user):,}명",
     "처리": "'unknown'으로 채움",
     "근거": "A/B 간 결측 비율 동일, 보조 컬럼이라 유저 삭제 시 표본만 감소"},
])

cleaning_log

,이슈,대상 테이블,규모,처리,근거
0,내부 테스트 계정,titles 제외 전체,200명,전 테이블에서 제거,"test_ 접두 ID, 체험 5분 내 결제·100코인 일괄 사용 등 비정상 행동"
1,중복 결제 이벤트,subscription_events,66건,event_id 외 동일 행 중 첫 행만 유지,동일 유저·시각·금액 결제가 2회 기록 (로그 중복)
2,노출 기록 유실,titles 제외 전체,78명,전 테이블에서 제거,"실험군(A/B) 확인 불가, 체험 시작자의 0.30%·결제액의 0.24%로 영향 미미"
3,유입 경로 결측,users,"6,388명",'unknown'으로 채움,"A/B 간 결측 비율 동일, 보조 컬럼이라 유저 삭제 시 표본만 감소"


In [13]:
# ── 정제 전 행 수: 원본 파일을 다시 읽어서 세기 ──
raw_files = {
    "users": "users.csv.gz",
    "exposures": "exposures.csv.gz",
    "subscription_events": "subscription_events.csv",
    "coin_transactions": "coin_transactions.csv.gz",
    "daily_reading": "daily_reading.csv.gz",
    "titles": "titles.csv",
}

# ── 테이블마다 정제 전/후 행 수를 한 줄씩 쌓기 ──
rows = []
for name, file in raw_files.items():
    before = len(pd.read_csv(DATA / file))   # 정제 전 (원본 파일)
    after = len(table_info[name])            # 정제 후 (딕셔너리 속 표)
    rows.append({
        "테이블": name,
        "정제 전": before,
        "정제 후": after,
        "제거": before - after,
        "제거율(%)": round((before - after) / before * 100, 2),
    })

row_summary = pd.DataFrame(rows)
row_summary

,테이블,정제 전,정제 후,제거,제거율(%)
0,users,315200,314922,278,0.09
1,exposures,315122,314922,200,0.06
2,subscription_events,61011,60367,644,1.06
3,coin_transactions,825552,823238,2314,0.28
4,daily_reading,1865128,1859866,5262,0.28
5,titles,6300,6300,0,0.00


In [14]:
# ── 전환율 계산 함수 ──
# 이벤트 표를 받아서 "결제 유저 수 ÷ 체험 유저 수"를 돌려줌 (고유 유저 기준)
def conversion_rate(events):
    paid = events.loc[events["event_type"] == "payment", "user_id"].nunique()
    trial = events.loc[events["event_type"] == "trial_start", "user_id"].nunique()
    return paid / trial * 100

raw_events = pd.read_csv(DATA / "subscription_events.csv")

print(f"정제 전 전환율: {conversion_rate(raw_events):.2f}%")
print(f"정제 후 전환율: {conversion_rate(table_info['subscription_events']):.2f}%")

정제 전 전환율: 29.53%
정제 후 전환율: 29.01%


In [15]:
# ── 개별 변수를 정제된 표로 맞추기 ──
users = table_info["users"]
exposures = table_info["exposures"]
subscription_events = table_info["subscription_events"]
coin_transactions = table_info["coin_transactions"]
daily_reading = table_info["daily_reading"]
titles = table_info["titles"]

In [16]:
# ── 정제 결과 저장 ──
CLEAN = DATA / "clean"
CLEAN.mkdir(exist_ok=True)     # 폴더가 없으면 만들고, 있으면 넘어감

for name, df in table_info.items():
    df.to_csv(CLEAN / f"{name}.csv.gz", index=False)
    print(f"{name}: {len(df):,}행 저장")

coin_transactions: 823,238행 저장
daily_reading: 1,859,866행 저장
exposures: 314,922행 저장
subscription_events: 60,367행 저장
titles: 6,300행 저장
users: 314,922행 저장
